# ENEC Enterprise AI Training
## A-001 Hybrid AI Agent: SQL + RAG + LLM

**Training scenario:** Synthetic enterprise asset reliability case  
**Asset:** A-001 — Cooling Water Pump

### Goal
Build a simple, governed AI assistant that combines:

- Structured data from Databricks SQL
- Unstructured evidence from A-001 PDF documents
- LLM reasoning
- Human review

> **Important:** This notebook is for synthetic training only. The AI supports human review. It must not diagnose equipment failure, authorize maintenance, close work orders, or control equipment.

## Learning Outcomes
By the end of this notebook, participants should be able to:

1. Explain why SQL and RAG solve different parts of the same enterprise problem.
2. Query structured asset data from Databricks.
3. Load and chunk PDF documents.
4. Create embeddings and perform semantic retrieval.
5. Combine SQL facts and document evidence.
6. Build a simple hybrid agent.
7. Apply governance rules around document authority and human approval.

## Architecture

```text
User Question
     |
     v
Hybrid A-001 Agent
     |
     +-------------------+
     |                   |
     v                   v
 SQL Tool            RAG Tool
     |                   |
 asset_360          A-001 PDFs
     |                   |
     +---------+---------+
               |
               v
              LLM
               |
               v
 Evidence-backed briefing
               |
               v
         Human Reviewer
```

# 1. Environment Setup

We keep the first version intentionally simple.

Libraries:
- `pypdf` for PDF text extraction
- `numpy` for vector similarity
- `mlflow.deployments` for Databricks model endpoints

We are deliberately not using LangChain yet. First understand:

**SQL → Retrieval → Evidence → LLM → Review**

In [ ]:
%pip install pypdf mlflow numpy

> **Checkpoint:** Restart Python if Databricks asks you to after installation.

# 2. Configuration

Suggested document location:

```text
/Volumes/main/default/enec_training/A001_RAG_Documents
```

Suggested asset table:

```text
main.default.asset_360
```

Update these values to match your workspace.

In [ ]:
from pathlib import Path
import numpy as np
from pypdf import PdfReader
from mlflow.deployments import get_deploy_client

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Day_2_GenAI_RAG").exists()), None)
DOCUMENT_FOLDER = str(REPO_ROOT / "Day_2_GenAI_RAG/documents/a001") if REPO_ROOT else "/Workspace/Nuclear_Enterprise_360/A001 Documents"
ASSET_TABLE = "workspace.nuclear_enterprise_360.asset_360"

EMBEDDING_MODEL = "databricks-qwen3-embedding-0-6b"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"

client = get_deploy_client("databricks")

print("Document folder :", DOCUMENT_FOLDER)
print("Asset table     :", ASSET_TABLE)
print("Embedding model :", EMBEDDING_MODEL)
print("Chat model      :", CHAT_MODEL)

# 3. Business Question

We are **not** asking:

> What repair should be performed?

We ask:

> **Does the available evidence for A-001 support escalation for qualified reliability review?**

The AI should collect evidence, identify approved guidance, highlight uncertainty, and recommend a human review.

It should not:
- diagnose failure,
- authorize repair,
- close work,
- issue equipment commands.

# 4. Structured Data with SQL

SQL is best for precise facts:
- health score,
- risk level,
- open work,
- high-priority work,
- findings.

In [ ]:
query = f'''
SELECT
    asset_id,
    asset_name,
    criticality,
    health_score,
    risk_level,
    open_work_orders,
    high_priority_open_work,
    follow_up_findings
FROM {ASSET_TABLE}
WHERE asset_id = 'A-001'
LIMIT 1
'''

display(spark.sql(query))

## Discussion

Ask:
1. Which fields are direct observations?
2. Which fields are calculated indicators?
3. Does a health score prove failure?
4. Does an open work order prove completion?

**Key point:** structured data can justify investigation, not automatic action.

# 5. Convert SQL Access into a Controlled Tool

The SQL tool:
- accepts an asset ID,
- runs a bounded query,
- is read-only,
- does not let the LLM generate arbitrary SQL.

In [ ]:
def sql_tool(asset_id="A-001"):
    safe_asset_id = asset_id.replace("'", "")

    query = f'''
    SELECT
        asset_id,
        asset_name,
        criticality,
        health_score,
        risk_level,
        open_work_orders,
        high_priority_open_work,
        follow_up_findings
    FROM {ASSET_TABLE}
    WHERE asset_id = '{safe_asset_id}'
    LIMIT 1
    '''

    rows = spark.sql(query).collect()

    if not rows:
        return {
            "query": query,
            "result": None,
            "message": f"No record found for {asset_id}"
        }

    return {
        "query": query,
        "result": rows[0].asDict()
    }

sql_evidence = sql_tool("A-001")
sql_evidence

# 6. Load the A-001 Document Corpus

The corpus contains multiple document types, including:
- enterprise reliability policy,
- operating guide,
- superseded procedure,
- current approved procedure,
- draft future procedure,
- troubleshooting guide,
- condition monitoring report,
- open inspection work order,
- field condition report.

This creates an important RAG lesson:

> **The most similar document is not always the authoritative document.**

In [ ]:
pdf_files = sorted(Path(DOCUMENT_FOLDER).glob("*.pdf"))

print("PDF files found:", len(pdf_files))

for p in pdf_files:
    print("-", p.name)

> **Expected:** 9 A-001 PDFs. If you see 0, check the folder path and confirm the ZIP contents were extracted.

# 7. Extract Text from PDFs

We preserve:
- filename,
- page number,
- text.

This allows the final answer to show where evidence came from.

In [ ]:
documents = []

for pdf_path in pdf_files:
    reader = PdfReader(str(pdf_path))

    for page_number, page in enumerate(reader.pages, start=1):
        text = page.extract_text()

        if text and text.strip():
            documents.append({
                "source": pdf_path.name,
                "page": page_number,
                "text": text.strip()
            })

print("Pages loaded:", len(documents))
documents[:2]

# 8. Chunk the Documents

For a simple teaching demo:
- chunk size ≈ 1000 characters
- overlap = 150 characters

Overlap helps preserve context across chunk boundaries.

In [ ]:
def chunk_text(text, chunk_size=1000, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap

    return chunks

chunks = []

for doc in documents:
    page_chunks = chunk_text(doc["text"])

    for chunk_id, chunk in enumerate(page_chunks):
        chunks.append({
            "source": doc["source"],
            "page": doc["page"],
            "chunk_id": chunk_id,
            "text": chunk
        })

print("Total chunks:", len(chunks))

# 9. Create Embeddings

An embedding converts text into a vector.

Semantically similar text should have vectors that are close together.

For this workshop, vectors are kept in Python memory for simplicity.

In [ ]:
def get_embedding(text):
    response = client.predict(
        endpoint=EMBEDDING_MODEL,
        inputs={"input": text}
    )

    return np.array(response["data"][0]["embedding"])

sample_vector = get_embedding("A-001 has rising vibration.")

print("Embedding dimensions:", len(sample_vector))
print("First 10 values:", sample_vector[:10])

> **Checkpoint:** the vector is not a readable summary. It is a numerical representation used for semantic comparison.

In [ ]:
for i, chunk in enumerate(chunks):
    chunk["embedding"] = get_embedding(chunk["text"])

    if i % 10 == 0:
        print(f"Embedded {i}/{len(chunks)}")

print("Embedding complete.")

# 10. Build the RAG Retrieval Tool

We use cosine similarity between:
- the user-question vector
- each document-chunk vector

We return:
- source,
- page,
- score,
- text.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def document_tool(question, top_k=5):
    question_embedding = get_embedding(question)

    results = []

    for chunk in chunks:
        score = cosine_similarity(
            question_embedding,
            chunk["embedding"]
        )

        results.append({
            "score": float(score),
            "source": chunk["source"],
            "page": chunk["page"],
            "text": chunk["text"]
        })

    results.sort(key=lambda x: x["score"], reverse=True)

    return results[:top_k]

# 11. Test Retrieval Before Building the Agent

Always inspect retrieval quality before trusting the final LLM answer.

In [ ]:
test_question = "Which inspection procedure is currently approved for A-001?"

retrieved = document_tool(test_question, top_k=5)

for i, item in enumerate(retrieved, start=1):
    print("=" * 80)
    print("RESULT :", i)
    print("SOURCE :", item["source"])
    print("PAGE   :", item["page"])
    print("SCORE  :", round(item["score"], 4))
    print()
    print(item["text"][:700])
    print()

## Governance Check

The corpus deliberately contains:
- **V1** — superseded
- **V2** — approved/current
- **V3D** — draft/newer but not approved

A naive RAG system may retrieve the newest or most similar version.

A governed system must understand:

**relevance ≠ authority**

# 12. Add a Simple Authority Label

For teaching, we infer authority from filenames.

In production, approval status should be stored as explicit metadata during ingestion.

In [ ]:
def authority_label(filename):
    name = filename.upper()

    if "DRAFT" in name or "V3D" in name:
        return "DRAFT"

    if "SUPERSEDED" in name or "V1_" in name:
        return "SUPERSEDED"

    if "WORK_ORDER" in name or "WO-" in name:
        return "OPEN_RECORD"

    return "APPROVED_OR_RECORD"

def governed_document_tool(question, top_k=6):
    raw_results = document_tool(question, top_k=top_k * 2)

    governed = []

    for item in raw_results:
        item = dict(item)
        item["authority"] = authority_label(item["source"])
        governed.append(item)

    return governed[:top_k]

In [ ]:
governed_results = governed_document_tool(
    "What procedure should be used to review the rising vibration trend for A-001?"
)

for item in governed_results:
    print(
        item["authority"],
        "|",
        item["source"],
        "| score:",
        round(item["score"], 3)
    )

# 13. LLM Helper

The LLM receives both evidence and explicit boundaries.

This is where we teach that a good enterprise agent is not just a model call — it is a model call surrounded by controls.

In [ ]:
def call_llm(prompt):
    response = client.predict(
        endpoint=CHAT_MODEL,
        inputs={
            "messages": [
                {
                    "role": "system",
                    "content": '''
You are an enterprise asset reliability decision-support assistant.

Use only the evidence supplied to you.

Rules:
1. Do not diagnose a physical equipment failure.
2. Do not authorize maintenance or repair.
3. Do not close or approve work orders.
4. Do not control equipment.
5. Clearly distinguish observed facts from interpretation.
6. Treat DRAFT and SUPERSEDED procedures as non-authoritative.
7. Preserve missing or conflicting evidence.
8. The final operational decision belongs to a qualified human reviewer.
'''
                },
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            "temperature": 0.1,
            "max_tokens": 1000
        }
    )

    return response["choices"][0]["message"]["content"]

# 14. Build the Hybrid A-001 Agent

The first version is transparent and deterministic:

1. Query SQL.
2. Retrieve documents.
3. Label document authority.
4. Assemble evidence.
5. Ask the LLM to write a bounded briefing.

In [ ]:
def a001_agent(question, asset_id="A-001"):

    # STEP 1 — Structured evidence
    sql_evidence = sql_tool(asset_id)

    # STEP 2 — Document evidence
    rag_evidence = governed_document_tool(
        question,
        top_k=6
    )

    # STEP 3 — Format document context
    document_context = ""

    for i, doc in enumerate(rag_evidence, start=1):
        document_context += f'''
DOCUMENT {i}
Source: {doc["source"]}
Page: {doc["page"]}
Authority: {doc["authority"]}
Similarity score: {doc["score"]:.4f}

{doc["text"]}

'''

    # STEP 4 — Final evidence prompt
    prompt = f'''
USER QUESTION
{question}

==================================================
STRUCTURED SQL EVIDENCE
==================================================

SQL Query:
{sql_evidence["query"]}

SQL Result:
{sql_evidence["result"]}

==================================================
DOCUMENT EVIDENCE
==================================================

{document_context}

==================================================
TASK
==================================================

Prepare an evidence-backed reliability review briefing.

Use exactly these sections:

1. Observed Facts
2. Trend / Condition Evidence
3. Applicable Approved Guidance
4. Work / Record Evidence
5. Assessment
6. Evidence Gaps or Uncertainty
7. Recommended Human Next Step
8. Sources Used

Important:
- Do not diagnose equipment failure.
- Do not authorize repair.
- Do not claim an open work order is completed.
- Do not use a DRAFT or SUPERSEDED procedure as current authority.
- If evidence conflicts, state the conflict.
'''

    final_answer = call_llm(prompt)

    return {
        "question": question,
        "sql_evidence": sql_evidence,
        "rag_evidence": rag_evidence,
        "answer": final_answer
    }

# 15. Main Workshop Demo

Run this question:

> **Investigate A-001. Does the available evidence support escalation for qualified reliability review? Explain why and identify the currently approved inspection procedure.**

In [ ]:
question = '''
Investigate A-001.

Does the available evidence support escalation for qualified reliability review?

Explain why and identify the currently approved inspection procedure.
'''

result = a001_agent(question)

print(result["answer"])

# 16. Inspect the Evidence

A responsible AI application should make evidence inspectable.

Do not evaluate only the final answer.

In [ ]:
print("SQL EVIDENCE")
print("=" * 80)
print(result["sql_evidence"])

print("\nRAG EVIDENCE")
print("=" * 80)

for item in result["rag_evidence"]:
    print(
        item["authority"],
        "|",
        item["source"],
        "| Page:",
        item["page"],
        "| Score:",
        round(item["score"], 4)
    )

# 17. Evaluation Questions

### Direct retrieval
1. What is A-001 used for?
2. Which signals are monitored for A-001?

### Version control
3. Which inspection procedure is currently approved?
4. Is version 3.0 the current procedure?
5. Why should version 1.0 not be used?

### Troubleshooting
6. What should be checked when vibration is rising?
7. Does one high vibration reading prove failure?

### Work records
8. What work is currently requested for A-001?
9. Does the open work order authorize component replacement?

### Hybrid SQL + RAG
10. What is the current health score of A-001 and which approved procedure applies?

### Governance
11. Who should make the final maintenance authorization?

# 18. Participant Exercise

Ask the agent:

> **A-001 has rising vibration. Summarize the structured evidence, identify the relevant approved guidance, and explain what additional evidence a qualified reviewer should inspect.**

The answer should include:
- SQL facts,
- trend evidence,
- current approved procedure,
- open-work evidence,
- uncertainty,
- human next step.

It should not include:
- failure diagnosis,
- repair order,
- work-order closure,
- equipment command.

In [ ]:
exercise_question = '''
A-001 has rising vibration.

Summarize the structured evidence,
identify the relevant approved guidance,
and explain what additional evidence
a qualified reviewer should inspect.
'''

exercise_result = a001_agent(exercise_question)

print(exercise_result["answer"])

# 19. Agent Skills + Progressive Disclosure

A useful next step is to stop giving the LLM every instruction and every tool description at once.

Instead, we introduce **Skills**.

A skill is a bounded capability with:

- a name,
- a short description,
- an input contract,
- detailed instructions,
- approved tools,
- output expectations,
- safety / authority limits.

## Why Progressive Disclosure?

The agent should not receive the full instructions for every capability on every turn.

Instead:

```text
LEVEL 1 — DISCOVER
Agent sees only:
Skill name + short description

        ↓ choose one

LEVEL 2 — LOAD
Agent receives:
Detailed instructions + approved tools

        ↓ execute

LEVEL 3 — USE
Agent calls only the tools required
for the selected skill
```

This reduces prompt size, improves tool selection, and makes governance easier.

### A-001 Skill Set

1. **asset_summary** — get current structured facts for A-001.
2. **approved_procedure** — identify the current authoritative procedure.
3. **reliability_review** — combine SQL + document evidence into a human-review briefing.

> **The agent sees capabilities first. It sees detailed instructions only after selecting a capability.**


In [ ]:
# LEVEL 1 — COMPACT SKILL CATALOG

SKILL_CATALOG = {
    "asset_summary": {
        "description": "Retrieve current structured asset facts using the governed SQL tool."
    },
    "approved_procedure": {
        "description": "Find the current approved procedure and reject draft/superseded guidance as authority."
    },
    "reliability_review": {
        "description": "Combine SQL facts and governed document evidence into a reliability review briefing."
    }
}

for skill_name, meta in SKILL_CATALOG.items():
    print(f"{skill_name:22} -> {meta['description']}")


## Level 1 — Discovery

At this point, the model sees only compact descriptions. It does **not** yet receive every detailed workflow instruction, tool rule, or output template.


In [ ]:
# LEVEL 2 — DETAILED SKILL DEFINITIONS
# Loaded only after a skill is selected.

SKILL_DETAILS = {
    "asset_summary": {
        "purpose": "Return the current structured condition summary for an asset.",
        "instructions": """
1. Use only the governed sql_tool.
2. Query only the requested asset.
3. Do not infer equipment failure from health_score or risk_level.
4. Distinguish calculated indicators from physical measurements.
5. Return a concise factual summary.
""",
        "allowed_tools": ["sql_tool"],
        "output_format": [
            "Asset ID", "Health score", "Risk level", "Open work",
            "High-priority open work", "Follow-up findings", "Interpretation boundary"
        ]
    },

    "approved_procedure": {
        "purpose": "Identify authoritative document guidance.",
        "instructions": """
1. Use the governed document retrieval tool.
2. Search for the relevant inspection procedure.
3. Prefer current APPROVED guidance.
4. Never treat DRAFT content as current authority.
5. Never treat SUPERSEDED content as current authority.
6. State the selected source and why it is authoritative.
""",
        "allowed_tools": ["governed_document_tool"],
        "output_format": [
            "Current approved procedure", "Status", "Version",
            "Relevant guidance", "Rejected non-authoritative sources"
        ]
    },

    "reliability_review": {
        "purpose": "Prepare an evidence-backed reliability review briefing.",
        "instructions": """
1. Call sql_tool for structured asset facts.
2. Call governed_document_tool for relevant document evidence.
3. Separate facts from interpretation.
4. Use APPROVED guidance as authority.
5. Treat OPEN work orders as requested work, not completed work.
6. Do not diagnose physical failure.
7. Do not authorize maintenance or repair.
8. Preserve uncertainty and conflicts.
9. End with a recommended human next step.
""",
        "allowed_tools": ["sql_tool", "governed_document_tool", "call_llm"],
        "output_format": [
            "Observed Facts", "Document Guidance", "Assessment",
            "Evidence Gaps", "Recommended Human Next Step", "Sources"
        ]
    }
}


def load_skill(skill_name):
    if skill_name not in SKILL_DETAILS:
        raise ValueError(f"Unknown skill: {skill_name}")
    return SKILL_DETAILS[skill_name]


## Level 2 — Load Only the Selected Skill

The detailed instructions become visible only after the skill has been chosen.


In [ ]:
selected_skill = "approved_procedure"
skill = load_skill(selected_skill)

print("SELECTED SKILL:", selected_skill)
print("PURPOSE:", skill["purpose"])
print("ALLOWED TOOLS:", skill["allowed_tools"])
print("INSTRUCTIONS:")
print(skill["instructions"])


# 20. Simple Skill Router

For teaching, start with a transparent rule-based router before introducing an LLM planner.

```text
asset condition / score / counts  → asset_summary
approved procedure / version      → approved_procedure
investigate / assess / escalate   → reliability_review
```


In [ ]:
def select_skill(question):
    q = question.lower()

    review_terms = [
        "investigate", "reliability review", "recommend",
        "assessment", "escalate", "evidence support", "should"
    ]
    if any(term in q for term in review_terms):
        return "reliability_review"

    procedure_terms = [
        "procedure", "approved", "guidance", "version", "document"
    ]
    if any(term in q for term in procedure_terms):
        return "approved_procedure"

    return "asset_summary"


test_questions = [
    "What is the current health score of A-001?",
    "Which inspection procedure is currently approved for A-001?",
    "Does the evidence support escalation for qualified reliability review?"
]

for q in test_questions:
    print(q)
    print(" ->", select_skill(q))
    print()


# 21. Execute Only the Selected Skill

This is the third layer of progressive disclosure: only the approved tools for that skill are used.


In [ ]:
def execute_asset_summary(question, asset_id="A-001"):
    evidence = sql_tool(asset_id)

    prompt = f"""
Question:
{question}

Structured evidence:
{evidence['result']}

Prepare a short factual asset summary.
Do not diagnose equipment failure.
Explain that health/risk values are analytical indicators.
"""
    return call_llm(prompt)


def execute_approved_procedure(question):
    evidence = governed_document_tool(question, top_k=6)
    context = ""

    for item in evidence:
        context += f"""
Source: {item['source']}
Page: {item['page']}
Authority: {item['authority']}
Text:
{item['text']}

"""

    prompt = f"""
Question:
{question}

Retrieved document evidence:
{context}

Identify the current authoritative procedure.
Rules:
- APPROVED guidance may be authoritative.
- DRAFT is not current authority.
- SUPERSEDED is historical only.
- Explain why the selected source is authoritative.
"""
    return call_llm(prompt)


def execute_reliability_review(question, asset_id="A-001"):
    return a001_agent(question, asset_id=asset_id)["answer"]


def execute_skill(skill_name, question, asset_id="A-001"):
    skill = load_skill(skill_name)

    print("Loaded skill:", skill_name)
    print("Allowed tools:", skill["allowed_tools"])
    print()

    if skill_name == "asset_summary":
        return execute_asset_summary(question, asset_id)
    if skill_name == "approved_procedure":
        return execute_approved_procedure(question)
    if skill_name == "reliability_review":
        return execute_reliability_review(question, asset_id)

    raise ValueError("Unsupported skill")


# 22. Progressive Disclosure Agent

```text
User Question
     |
     v
Compact Skill Catalog
     |
     v
Skill Router
     |
     v
Load ONE Skill
     |
     v
Approved Tools Only
     |
     v
Evidence-backed Output
```


In [ ]:
def progressive_agent(question, asset_id="A-001"):
    print("QUESTION")
    print(question)
    print("=" * 80)

    # LEVEL 1 — discover/select
    selected_skill = select_skill(question)
    print("Selected skill:", selected_skill)

    # LEVEL 2 + 3 — load and execute only selected skill
    answer = execute_skill(
        selected_skill,
        question,
        asset_id=asset_id
    )

    return {
        "selected_skill": selected_skill,
        "answer": answer
    }


# 23. Demo the Progressive Agent

Try different question types and observe which skill is loaded.


In [ ]:
questions = [
    "What is the current condition summary for A-001?",
    "Which inspection procedure is currently approved for A-001?",
    "Investigate A-001. Does the available evidence support escalation for qualified reliability review?"
]

for question in questions:
    result = progressive_agent(question)

    print("\nANSWER")
    print(result["answer"])
    print("\n" + "#" * 100 + "\n")


# 24. Why Progressive Disclosure Helps

Without progressive disclosure, every request can carry every tool description, every workflow rule, and every output template.

With progressive disclosure:

```text
Question
   ↓
small skill catalog
   ↓
select skill
   ↓
load only relevant instructions
   ↓
use only relevant tools
```

### Benefits

- **Lower context usage** — load only relevant instructions.
- **Better routing** — clear capability boundaries.
- **Better governance** — each skill has explicit allowed tools.
- **Independent testing** — evaluate each skill separately.
- **Easier maintenance** — update one skill without rewriting the full agent.

> **Enterprise design principle:** expose capability gradually based on what the current task actually requires.


# 25. What Makes This an Agent?

This system has:

### Goal
Investigate an A-001 reliability question.

### Tools
- SQL tool
- RAG/document tool

### Reasoning
The LLM combines evidence from different sources.

### Constraints
The system cannot authorize maintenance and cannot treat non-approved documents as current guidance.

### Output
A review briefing for a human.

This is a **controlled decision-support agent**.

It is not yet a fully autonomous agent.

# 26. Next Step: Day 4 Agentic AI

```text
User Goal
    |
    v
Policy Guard
    |
    v
Tool Router
   / \
SQL RAG
   \ /
 Evidence
    |
    v
Draft Recommendation
    |
    v
Human Approval
    |
    v
Audit Trail
```

Next capabilities:
- automatic tool selection,
- state tracking,
- human approval gates,
- refusal rules,
- audit logging,
- replayable traces.

> **Agentic AI is controlled orchestration, not unrestricted autonomy.**

# 27. Key Takeaways

### SQL
Best for exact structured facts.

### RAG
Best for retrieving document knowledge and contextual guidance.

### LLM
Best for synthesizing evidence into a readable briefing.

### Agent
Coordinates tools toward a goal.

### Governance
Defines what the agent is allowed to do.

## Final Workshop Message

```text
SQL tells us WHAT the data says.
RAG tells us WHAT the documents say.
The LLM helps explain WHAT IT MEANS.
A qualified human decides WHAT TO DO.
```